# Workspace RayCluster + job client — Ray Data

Topic 1 uses the CodeFlare **workspace cluster** pattern that works reliably from an OpenShift AI workbench:

1. Authenticate to the OpenShift API
2. List LocalQueues with `list_local_queues()`
3. Attach or create the shared `ray-workshop` RayCluster (`ensure_workshop_cluster`)
4. Submit work with `cluster.job_client` (Ray Jobs API on the head)
5. Inspect with `view_clusters()` (and Ray Dashboard → Jobs)
6. **Leave the cluster up** for Topics 2–3 (tear down at the end of Topic 3)

Runs `scale_data.py` on KubeRay workers.

Official docs: [Running Ray workloads from Jupyter](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.4/html/working_with_distributed_workloads/running-ray-based-distributed-workloads_distributed-workloads)


In [ ]:
# JupyterLab cwd is usually extras/notebooks — find the repo for working_dir.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Get **server** and **token** from the OpenShift Console (not from `oc whoami` inside the workbench):

1. OpenShift Console → your username → **Copy login command** → Display token
2. Paste values below

Lab clusters with self-signed API certs need `verify_ssl=False`.

In [ ]:
from kube_authkit import AuthConfig, get_k8s_client
from codeflare_sdk import set_api_client, list_local_queues, view_clusters

# Console → your username → Copy login command → Display token
OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth_config = AuthConfig(
    method="openshift",
    k8s_api_host=OPENSHIFT_SERVER.strip(),
    token=OPENSHIFT_TOKEN.strip(),
    verify_ssl=False,  # lab/self-signed API certs
)
set_api_client(get_k8s_client(config=auth_config))

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)


## Attach or create the RayCluster

Shared name **`ray-workshop`** (2 workers × 1 GPU). Reused by Topics 2–3 — do not tear down unless you are stopping for the day.


In [ ]:
from workshop_bootstrap import ensure_workshop_cluster

cluster = ensure_workshop_cluster(namespace=NAMESPACE, local_queue=LOCAL_QUEUE)
cluster.details()


## Submit with `job_client`

Packages the repo as `working_dir` and runs `scale_data.py` on the cluster.

In [ ]:
import time

client = cluster.job_client

submission_id = client.submit_job(
    entrypoint="python extras/scripts/scale_data.py",
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "pip": ["pyarrow>=14", "pandas>=2"],
        "env_vars": {
            "INPUT_PATH": "extras/data/iris.csv",
            "OUTPUT_DIR": "/tmp/ray-workshop-output/iris",
        },
    },
)
print(f"Submitted: {submission_id}")

terminal = {"SUCCEEDED", "FAILED", "STOPPED"}
deadline = time.time() + 900

while time.time() < deadline:
    status = client.get_job_status(submission_id)
    print(f"Job {submission_id}: {status}")
    if str(status).split(".")[-1].upper() in terminal:
        break
    time.sleep(15)
else:
    raise TimeoutError(f"Timed out waiting for job {submission_id}")

print(client.get_job_logs(submission_id))

## Observe

`view_clusters()` shows RayClusters in the project. Open the Ray Dashboard link → **Jobs** to see `job_client` submissions.

Optional CLI:

```sh
oc get raycluster,pods -n ray-workshop
oc describe localqueue ray-workshop-queue -n ray-workshop
```


In [ ]:
view_clusters(NAMESPACE)


## Tear down (optional)

**Prefer leaving the cluster up** for Topics 2–3. Run the next cell only if you are done for the day or the cluster is stuck. Topic 3 tears down by default.


In [ ]:
# Optional — skip to keep the cluster for Topics 2–3
# cluster.down()
# print("Cluster down.")
print("Leaving cluster up for Topics 2–3. Uncomment cluster.down() above if needed.")


## Verify

1. Job logs contain `Done. Wrote N parquet file(s)`.
2. `list_local_queues()` showed `ray-workshop-queue`.
3. `view_clusters()` showed `ray-workshop` while the cluster was up.

Jobs submitted via `job_client` appear in the **Ray Dashboard → Jobs** tab, not always as Kubernetes `RayJob` CRs.
